# ST1630 — Comparativa empírica SQL (PostgreSQL) vs NoSQL (MongoDB)

**Persona 4** — Consultas operacionales, réplica de recomendaciones Gold y mediciones reproducibles.

**Dependencias (entorno local):** `pip install pymongo psycopg2-binary pandas matplotlib`

## Protocolo
- **10 repeticiones** (tras **2 warm-ups** cuando aplica) para latencias y para **throughput** de escritura masiva.
- Se reporta **media** y **desviación estándar** (latencias en ms; throughput en filas/s).
- Servicios: `docker-compose up -d` (MongoDB `localhost:27017`, PostgreSQL `localhost:5432`).

## Datos
- **Recomendaciones:** si existe `data/gold_recommendations.csv` (exportadas desde Spark/Gold), se usa ese archivo; si no, se generan filas sintéticas con la misma forma `(user_id, movie_id, predicted_score)` para poder ejecutar el cuaderno sin MinIO.
- **Actividad por género:** se intenta espejar `genre_activity` desde Mongo (Flink); si la colección está vacía, se inserta una muestra sintética en ambos sistemas solo para el benchmark de agregación.


In [1]:
# Configuración e imports
from __future__ import annotations

print("Arrancando celda 1…", flush=True)

import os
import sys
import time
from pathlib import Path
from statistics import mean, stdev
from typing import List, Sequence, Tuple

# El kernel DEBE tener estos paquetes. Si falla, en una terminal ejecutá exactamente:
#   "%PYTHON%" -m pip install matplotlib pandas psycopg2-binary pymongo
# donde %PYTHON% es el intérprete que elegiste arriba a la derecha en el notebook.
try:
    import matplotlib
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"No está instalado 'matplotlib' en el Python del kernel:\n  {sys.executable}\n\n"
        "Solución: en terminal -> python -m pip install matplotlib pandas psycopg2-binary pymongo\n"
        "O en Cursor: seleccioná el intérprete donde ya instalaste esos paquetes."
    ) from e

# Evita bloqueos del kernel en VS Code/Cursor/Windows: backend sin ventana GUI.
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import pandas as pd
    import psycopg2
    from psycopg2.extras import execute_batch
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"Falta una librería en: {sys.executable}\n"
        "Ejecutá: python -m pip install pandas psycopg2-binary"
    ) from e

print("Librerías base OK…", flush=True)


def _find_project_root() -> Path:
    """Encuentra la carpeta que contiene queries/mongo_queries.py (desde cwd o padres)."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "queries" / "mongo_queries.py").is_file():
            return base
        nested = base / "Recomendacion-Streaming"
        if (nested / "queries" / "mongo_queries.py").is_file():
            return nested
    raise FileNotFoundError(
        "No se encontró queries/mongo_queries.py. "
        "Abrí como carpeta el repo Recomendacion-Streaming o ejecutá Jupyter con cwd ahí."
    )


ROOT = _find_project_root()

QUERIES_DIR = ROOT / "queries"
if str(QUERIES_DIR) not in sys.path:
    sys.path.insert(0, str(QUERIES_DIR))

print("Importando mongo_queries…", flush=True)
from mongo_queries import (
    DEFAULT_DB_NAME,
    DEFAULT_MONGO_URI,
    benchmark_latencies,
    bulk_insert_recommendations_normalized,
    ensure_operational_indexes,
    get_db,
    mirror_genre_activity_to_list,
    q_anomaly_alerts_for_user,
    q_genre_most_active_in_recent_windows,
    q_recommendations_for_user,
    q_trending_top_k_latest_window,
    replace_gold_recommendations,
)

REPETITIONS = 10
WARMUP = 2

MONGO_URI = os.getenv("MONGO_URI", DEFAULT_MONGO_URI)
PG_CONF = dict(
    host=os.getenv("PGHOST", "localhost"),
    port=int(os.getenv("PGPORT", "5432")),
    dbname=os.getenv("PGDATABASE", "movielens"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", "postgres123"),
)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

print("Imports OK. ROOT =", ROOT)
print("Si la siguiente celda tarda ~5s y falla: levantá Mongo con docker compose.")


Arrancando celda 1…
Librerías base OK…
Importando mongo_queries…
Imports OK. ROOT = C:\Almacen\Cosas de la U\2026-1\Sis Intensivos\Proyecto Final\Recomendacion-Streaming
Si la siguiente celda tarda ~5s y falla: levantá Mongo con docker compose.


## 1) Consultas operacionales (MongoDB — mínimo 3)

Usamos las colecciones alimentadas por Flink: `trending_movies`, `genre_activity`, `anomaly_alerts`. Las funciones viven en `queries/mongo_queries.py`.


In [3]:
print("Conectando a MongoDB…")
mongo_db = get_db(MONGO_URI, DEFAULT_DB_NAME)
print("Creando/verificando índices…")
ensure_operational_indexes(mongo_db)

print("Ejecutando consultas operacionales…")
trending = q_trending_top_k_latest_window(mongo_db, k=5)
genres = q_genre_most_active_in_recent_windows(mongo_db, last_n_windows=20)
sample_user = 42
alerts = q_anomaly_alerts_for_user(mongo_db, user_id=sample_user, limit=10)

print("1) Top trending (última ventana):", len(trending), "filas (mostrar 3)")
print(trending[:3])
print("\n2) Géneros más activos (agg):", len(genres), "filas (mostrar 3)")
print(genres[:3])
print(f"\n3) Alertas para user_id={sample_user}:", len(alerts))
print(alerts[:3])


Conectando a MongoDB…
Creando/verificando índices…
Ejecutando consultas operacionales…
1) Top trending (última ventana): 0 filas (mostrar 3)
[]

2) Géneros más activos (agg): 0 filas (mostrar 3)
[]

3) Alertas para user_id=42: 0
[]


## 2) Esquema PostgreSQL (`gold`) y réplica de recomendaciones

- Tabla relacional normalizada `gold.user_recommendations` con **PK compuesta** `(user_id, movie_id)`.
- En Mongo se materializa **un documento por usuario** en `gold_user_recommendations` (patrón operacional típico NoSQL).
- Fuente: `data/gold_recommendations.csv` si existe (export CSV desde Spark/Iceberg); si no, datos sintéticos reproducibles (`seed=42`).


In [4]:
def pg_connect():
    # connect_timeout en segundos: evita colgar el kernel si Postgres no está arriba
    return psycopg2.connect(**PG_CONF, connect_timeout=10)


def init_gold_schema(cn) -> None:
    stmts = [
        "CREATE SCHEMA IF NOT EXISTS gold;",
        """
        CREATE TABLE IF NOT EXISTS gold.user_recommendations (
            user_id INT NOT NULL,
            movie_id INT NOT NULL,
            predicted_score DOUBLE PRECISION NOT NULL,
            PRIMARY KEY (user_id, movie_id)
        );
        """,
        "CREATE INDEX IF NOT EXISTS idx_rec_movie ON gold.user_recommendations(movie_id);",
        """
        CREATE TABLE IF NOT EXISTS gold.dim_movie (
            movie_id INT PRIMARY KEY,
            title TEXT NOT NULL,
            genres TEXT
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS gold.genre_activity_mirror (
            window_start TEXT NOT NULL,
            window_end TEXT NOT NULL,
            genre TEXT NOT NULL,
            event_count BIGINT NOT NULL
        );
        """,
        "CREATE INDEX IF NOT EXISTS idx_gap_mirror_genre ON gold.genre_activity_mirror(genre);",
        "CREATE INDEX IF NOT EXISTS idx_gap_mirror_end ON gold.genre_activity_mirror(window_end);",
    ]
    with cn.cursor() as cur:
        for s in stmts:
            cur.execute(s)
    cn.commit()


def load_recommendation_rows() -> List[Tuple[int, int, float]]:
    """CSV opcional o conjunto sintético acotado para benchmarks locales."""
    csv_path = ROOT / "data" / "gold_recommendations.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        cmap = {c.lower().replace(" ", "_"): c for c in df.columns}
        ucol = cmap.get("userid") or cmap.get("user_id")
        mcol = cmap.get("movieid") or cmap.get("movie_id")
        scol = cmap.get("predicted_score") or cmap.get("prediction") or cmap.get("score")
        if not (ucol and mcol and scol):
            raise ValueError(f"Columnas inválidas en {csv_path}: {df.columns.tolist()}")
        return list(
            zip(
                df[ucol].astype(int),
                df[mcol].astype(int),
                df[scol].astype(float),
            )
        )

    import random

    random.seed(42)
    n_users, k = 4000, 10
    rows: List[Tuple[int, int, float]] = []
    for u in range(1, n_users + 1):
        mids = random.sample(range(1, 8000), k)
        for m in mids:
            rows.append((u, m, round(random.uniform(1.0, 5.0), 4)))
    return rows


def pg_reload_recommendations(cn, rows: Sequence[Tuple[int, int, float]]) -> None:
    with cn.cursor() as cur:
        cur.execute("TRUNCATE gold.user_recommendations;")
    cn.commit()
    with cn.cursor() as cur:
        execute_batch(
            cur,
            "INSERT INTO gold.user_recommendations (user_id, movie_id, predicted_score) VALUES (%s,%s,%s)",
            list(rows),
            page_size=5000,
        )
    cn.commit()


def load_dim_movies_for_join(cn, movie_ids: Sequence[int]) -> None:
    """Carga mínima de dimensión título/género para el experimento JOIN."""
    movies_path = ROOT / "data" / "movies.csv"
    rows: List[Tuple[int, str, str]] = []
    if movies_path.exists():
        df = pd.read_csv(movies_path)
        col_mid = "movieId" if "movieId" in df.columns else df.columns[0]
        col_title = "title" if "title" in df.columns else df.columns[1]
        col_gen = "genres" if "genres" in df.columns else df.columns[2]
        want = set(int(x) for x in movie_ids)
        sub = df[df[col_mid].isin(want)]
        rows = list(
            zip(
                sub[col_mid].astype(int),
                sub[col_title].astype(str),
                sub[col_gen].astype(str),
            )
        )
    else:
        import random

        random.seed(7)
        for mid in movie_ids:
            rows.append((int(mid), f"Movie {mid}", "Drama|Comedy"))

    with cn.cursor() as cur:
        cur.execute("TRUNCATE gold.dim_movie;")
    cn.commit()
    with cn.cursor() as cur:
        execute_batch(
            cur,
            "INSERT INTO gold.dim_movie (movie_id, title, genres) VALUES (%s,%s,%s)",
            rows,
            page_size=2000,
        )
    cn.commit()


print("Conectando a PostgreSQL (hasta 10s si el servicio no responde)…")
cn = pg_connect()
print("PostgreSQL OK. Creando esquema gold si no existe…")
init_gold_schema(cn)

print("Cargando filas de recomendaciones (CSV o sintético)…")
all_rows = load_recommendation_rows()
pg_reload_recommendations(cn, all_rows)
replace_gold_recommendations(mongo_db, all_rows)

sample_user_id = int(all_rows[0][0])
movie_id_set = {r[1] for r in all_rows[:5000]}
load_dim_movies_for_join(cn, list(movie_id_set)[:4000])

# Mongo: misma dimensión para $lookup (JOIN lógico)
mongo_db["gold_dim_movie"].drop()
with cn.cursor() as cur:
    cur.execute("SELECT movie_id, title, genres FROM gold.dim_movie")
    dim_rows = cur.fetchall()
if dim_rows:
    mongo_db["gold_dim_movie"].insert_many(
        [{"movie_id": r[0], "title": r[1], "genres": r[2]} for r in dim_rows]
    )
    mongo_db["gold_dim_movie"].create_index("movie_id", unique=True)

print(f"Filas en réplica: {len(all_rows):,} | user_id de prueba PK: {sample_user_id}")
print("Mongo (PK):", q_recommendations_for_user(mongo_db, sample_user_id) is not None)


Conectando a PostgreSQL (hasta 10s si el servicio no responde)…
PostgreSQL OK. Creando esquema gold si no existe…
Cargando filas de recomendaciones (CSV o sintético)…
Filas en réplica: 40,000 | user_id de prueba PK: 1
Mongo (PK): True


## 3) Benchmarks (10 repeticiones + media ± desv. estándar)

1. **Lectura por clave lógica** (`user_id`): Mongo (documento embebido) vs PostgreSQL (`WHERE user_id = …`).
2. **Filtro + agregación** (`SUM` por `genre` en ventanas recientes): pipeline Mongo vs `GROUP BY` SQL sobre el mismo conjunto espejado.
3. **Throughput de escritura masiva**: `insert_many` por lotes vs `execute_batch` — **10 corridas** midiendo filas/s; se reporta media ± σ.
4. **JOIN**: `$lookup` + `$unwind` vs `JOIN` relacional recomendaciones–dimensión título.

> Tras el experimento (3) se **restaura** la réplica Gold para no dejar las tablas vacías.


In [5]:
print(">>> Benchmarks: esta celda suele tardar VARIOS MINUTOS (latencias + 10 escrituras masivas).")
print(">>> Esperá al menos 3–10 min en PC modestas antes de interrumpir el kernel.\n")

from pymongo import ASCENDING, DESCENDING


def build_genre_bench_docs():
    docs = mirror_genre_activity_to_list(mongo_db, limit_docs=25_000)
    if len(docs) >= 50:
        return docs
    import random
    from datetime import datetime, timedelta, timezone

    random.seed(99)
    out = []
    base = datetime.now(timezone.utc).replace(microsecond=0)
    for i in range(9000):
        end = base + timedelta(minutes=i)
        out.append(
            {
                "window_start": (end - timedelta(minutes=10)).isoformat().replace("+00:00", "Z"),
                "window_end": end.isoformat().replace("+00:00", "Z"),
                "genre": random.choice(["Action", "Drama", "Comedy", "Sci-Fi", "Horror"]),
                "event_count": random.randint(1, 120),
            }
        )
    return out


bench_docs = build_genre_bench_docs()
ends_sorted = sorted({str(d["window_end"]) for d in bench_docs}, reverse=True)[:25]

with cn.cursor() as cur:
    cur.execute("TRUNCATE gold.genre_activity_mirror;")
cn.commit()
with cn.cursor() as cur:
    rows_pg = [
        (str(d["window_start"]), str(d["window_end"]), str(d["genre"]), int(d["event_count"]))
        for d in bench_docs
    ]
    execute_batch(
        cur,
        "INSERT INTO gold.genre_activity_mirror (window_start, window_end, genre, event_count) VALUES (%s,%s,%s,%s)",
        rows_pg,
        page_size=4000,
    )
cn.commit()

mongo_db["persona4_genre_bench"].drop()
if bench_docs:
    mongo_db["persona4_genre_bench"].insert_many(bench_docs)
    mongo_db["persona4_genre_bench"].create_index(
        [("window_end", DESCENDING), ("genre", ASCENDING)]
    )

c_bench = mongo_db["persona4_genre_bench"]


def run_mongo_agg():
    pipeline = [
        {"$match": {"window_end": {"$in": ends_sorted}}},
        {"$group": {"_id": "$genre", "total_events": {"$sum": "$event_count"}}},
        {"$sort": {"total_events": -1}},
    ]
    list(c_bench.aggregate(pipeline))


def run_pg_agg():
    with cn.cursor() as cur:
        cur.execute(
            """
            SELECT genre, SUM(event_count) AS total_events
            FROM gold.genre_activity_mirror
            WHERE window_end = ANY(%s)
            GROUP BY genre
            ORDER BY total_events DESC;
            """,
            (ends_sorted,),
        )
        cur.fetchall()


def run_mongo_pk():
    q_recommendations_for_user(mongo_db, sample_user_id)


def run_pg_pk():
    with cn.cursor() as cur:
        cur.execute(
            "SELECT movie_id, predicted_score FROM gold.user_recommendations WHERE user_id = %s;",
            (sample_user_id,),
        )
        cur.fetchall()


def run_mongo_join():
    pipeline = [
        {"$match": {"user_id": sample_user_id}},
        {"$unwind": "$items"},
        {
            "$lookup": {
                "from": "gold_dim_movie",
                "localField": "items.movie_id",
                "foreignField": "movie_id",
                "as": "mv",
            }
        },
        {"$unwind": "$mv"},
        {"$project": {"_id": 0, "title": "$mv.title", "predicted_score": "$items.predicted_score"}},
    ]
    list(mongo_db["gold_user_recommendations"].aggregate(pipeline))


def run_pg_join():
    with cn.cursor() as cur:
        cur.execute(
            """
            SELECT m.title, r.predicted_score
            FROM gold.user_recommendations r
            JOIN gold.dim_movie m ON r.movie_id = m.movie_id
            WHERE r.user_id = %s;
            """,
            (sample_user_id,),
        )
        cur.fetchall()


pk_mongo = benchmark_latencies(run_mongo_pk, repetitions=REPETITIONS, warmup=WARMUP)
pk_pg = benchmark_latencies(run_pg_pk, repetitions=REPETITIONS, warmup=WARMUP)

agg_mongo = benchmark_latencies(run_mongo_agg, repetitions=REPETITIONS, warmup=WARMUP)
agg_pg = benchmark_latencies(run_pg_agg, repetitions=REPETITIONS, warmup=WARMUP)

join_mongo = benchmark_latencies(run_mongo_join, repetitions=REPETITIONS, warmup=WARMUP)
join_pg = benchmark_latencies(run_pg_join, repetitions=REPETITIONS, warmup=WARMUP)

import random

random.seed(123)
bulk_rows = []
for u in range(1, 2501):
    mids = random.sample(range(10_000, 40_000), 8)
    for m in mids:
        bulk_rows.append((u, m, round(random.uniform(1.0, 5.0), 4)))

rows_bulk = len(bulk_rows)

mongo_rates: list[float] = []
for _ in range(REPETITIONS):
    dt = bulk_insert_recommendations_normalized(mongo_db, bulk_rows, batch_size=1500)
    mongo_rates.append(rows_bulk / dt)

pg_rates: list[float] = []
for _ in range(REPETITIONS):
    t0 = time.perf_counter()
    with cn.cursor() as cur:
        cur.execute("TRUNCATE gold.user_recommendations;")
    cn.commit()
    with cn.cursor() as cur:
        execute_batch(
            cur,
            "INSERT INTO gold.user_recommendations (user_id, movie_id, predicted_score) VALUES (%s,%s,%s)",
            bulk_rows,
            page_size=2500,
        )
    cn.commit()
    pg_rates.append(rows_bulk / (time.perf_counter() - t0))

throughput_mongo = mean(mongo_rates)
sigma_tp_mongo = stdev(mongo_rates) if len(mongo_rates) > 1 else 0.0
throughput_pg = mean(pg_rates)
sigma_tp_pg = stdev(pg_rates) if len(pg_rates) > 1 else 0.0

# Restaurar réplica principal
pg_reload_recommendations(cn, all_rows)
replace_gold_recommendations(mongo_db, all_rows)
with cn.cursor() as cur:
    cur.execute("SELECT movie_id, title, genres FROM gold.dim_movie")
    dim_rows = cur.fetchall()
mongo_db["gold_dim_movie"].drop()
if dim_rows:
    mongo_db["gold_dim_movie"].insert_many(
        [{"movie_id": r[0], "title": r[1], "genres": r[2]} for r in dim_rows]
    )
    mongo_db["gold_dim_movie"].create_index("movie_id", unique=True)

latency_summary = pd.DataFrame(
    [
        {
            "Experimento": "Lectura por user_id (clave lógica)",
            "Mongo_media_ms": pk_mongo["mean_ms"],
            "Mongo_sigma_ms": pk_mongo["stdev_ms"],
            "Postgres_media_ms": pk_pg["mean_ms"],
            "Postgres_sigma_ms": pk_pg["stdev_ms"],
        },
        {
            "Experimento": "Filtro + SUM por género",
            "Mongo_media_ms": agg_mongo["mean_ms"],
            "Mongo_sigma_ms": agg_mongo["stdev_ms"],
            "Postgres_media_ms": agg_pg["mean_ms"],
            "Postgres_sigma_ms": agg_pg["stdev_ms"],
        },
        {
            "Experimento": "JOIN recomendaciones ↔ dim película",
            "Mongo_media_ms": join_mongo["mean_ms"],
            "Mongo_sigma_ms": join_mongo["stdev_ms"],
            "Postgres_media_ms": join_pg["mean_ms"],
            "Postgres_sigma_ms": join_pg["stdev_ms"],
        },
    ]
)

throughput_summary = pd.DataFrame(
    [
        {
            "Motor": "MongoDB (insert_many por lotes)",
            "Filas": rows_bulk,
            "Filas_por_segundo_media": throughput_mongo,
            "Filas_por_segundo_sigma": sigma_tp_mongo,
        },
        {
            "Motor": "PostgreSQL (execute_batch)",
            "Filas": rows_bulk,
            "Filas_por_segundo_media": throughput_pg,
            "Filas_por_segundo_sigma": sigma_tp_pg,
        },
    ]
)

latency_summary, throughput_summary


>>> Benchmarks: esta celda suele tardar VARIOS MINUTOS (latencias + 10 escrituras masivas).
>>> Esperá al menos 3–10 min en PC modestas antes de interrumpir el kernel.



(                           Experimento  Mongo_media_ms  Mongo_sigma_ms  \
 0   Lectura por user_id (clave lógica)         1.46287        0.311491   
 1              Filtro + SUM por género         1.66290        0.211894   
 2  JOIN recomendaciones ↔ dim película         1.76127        0.269282   
 
    Postgres_media_ms  Postgres_sigma_ms  
 0            0.36387           0.066268  
 1            1.32644           0.152468  
 2            0.69992           0.143697  ,
                              Motor  Filas  Filas_por_segundo_media  \
 0  MongoDB (insert_many por lotes)  20000            102993.195101   
 1       PostgreSQL (execute_batch)  20000             43598.169895   
 
    Filas_por_segundo_sigma  
 0             13225.537098  
 1              4577.464544  )

In [6]:
labels = ["PK\n(user_id)", "Agregación\n(género)", "JOIN\n(reco × peli)"]
x = range(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(
    [i - width / 2 for i in x],
    [pk_mongo["mean_ms"], agg_mongo["mean_ms"], join_mongo["mean_ms"]],
    width,
    yerr=[pk_mongo["stdev_ms"], agg_mongo["stdev_ms"], join_mongo["stdev_ms"]],
    label="MongoDB",
    capsize=3,
)
ax.bar(
    [i + width / 2 for i in x],
    [pk_pg["mean_ms"], agg_pg["mean_ms"], join_pg["mean_ms"]],
    width,
    yerr=[pk_pg["stdev_ms"], agg_pg["stdev_ms"], join_pg["stdev_ms"]],
    label="PostgreSQL",
    capsize=3,
)
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel("Latencia media (ms)")
ax.set_title("Comparativa de latencias (media ± σ, n=10)")
ax.legend()
plt.tight_layout()

fig2, ax2 = plt.subplots(figsize=(5, 3))
ax2.bar(
    ["MongoDB", "PostgreSQL"],
    [throughput_mongo, throughput_pg],
    yerr=[sigma_tp_mongo, sigma_tp_pg],
    capsize=4,
    color=["#5c9ead", "#e2847a"],
)
ax2.set_ylabel("Filas / segundo")
ax2.set_title("Throughput escritura masiva (media ± σ, n=10)")
plt.tight_layout()


## 4) Cómo interpretar y cómo exportar Gold a CSV

- **PK / documento embebido:** si Mongo entrega **un documento por usuario** (`gold_user_recommendations`), el patrón `find_one` suele minimizar round-trips frente a un bloque relacional de varias filas (aunque PostgreSQL con índice en `user_id` sea muy competitivo).
- **Agregaciones con filtros:** cuando el trabajo es **grupo + orden + sumas** sobre muchas filas, el optimizador SQL suele ser fuerte; en Mongo hay que diseñar bien índices y pipelines.
- **JOIN:** PostgreSQL optimiza `JOIN` entre tablas normalizadas; en Mongo el equivalente (`$lookup`) suele ser más costoso si hay muchos documentos intermedios (`$unwind`).
- **Throughput:** los resultados dependen de `batch_size`, red Docker y disco; documentá versión de Docker, RAM y si el benchmark corre en el host o dentro de un contenedor.

**Export rápido desde Spark (Jupyter del contenedor `spark-master`):**

```python
spark.table("local.gold.recommendations").coalesce(1) \
    .write.mode("overwrite").option("header", True) \
    .csv("file:/home/jovyan/data/gold_recommendations_csv")
```

Luego copiá el CSV resultante (part-00000...) a `Recomendacion-Streaming/data/gold_recommendations.csv` en tu máquina, o unificá el archivo en uno solo con cabecera `userId,movieId,predicted_score`.
